# الفصل 7: التمثيلات المتجهية للكلمات: Word2Vec وGloVe

> بعد Tokenization، لدينا تسلسل من الأعداد الصحيحة (Token IDs) — لكن هذه الأعداد عشوائية ولا معنى دلاليًا لها (الرمز 42 ليس "أقرب" للرمز 43 من الرمز 9000 بأي معنى رياضي مفيد).

- **الجزء:** 2 — التمثيل اللغوي
- **الفصل:** 7
- **المستوى:** متوسط
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 6: التقطيع إلى وحدات: Tokenization
- **الوسوم:** Word2Vec، GloVe، Negative Sampling، Embeddings

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-02-language-representation%2Fch07-word-embeddings.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-02-language-representation%2Fch07-word-embeddings.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch07-word-embeddings.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> بعد Tokenization، لدينا تسلسل من الأعداد الصحيحة (Token IDs) — لكن هذه الأعداد عشوائية ولا معنى دلاليًا لها (الرمز 42 ليس "أقرب" للرمز 43 من الرمز 9000 بأي معنى رياضي مفيد).

> **بيانات الفصل**
>
> - **الجزء:** 2 — التمثيل اللغوي
> - **المستوى:** متوسط
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 6: التقطيع إلى وحدات: Tokenization


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم GloVe — نهج إحصائي بديل
- فهم Static مقابل Contextual Embeddings — التوضيح الحاسم
- فهم Word2Vec


## 1. لماذا يهمّك هذا الموضوع؟

بعد Tokenization، لدينا تسلسل من الأعداد الصحيحة (Token IDs) — لكن هذه الأعداد **عشوائية ولا معنى دلاليًا لها** (الرمز 42 ليس "أقرب" للرمز 43 من الرمز 9000 بأي معنى رياضي مفيد). Word Embeddings تحلّ هذه المشكلة جذريًا: نُمثِّل كل رمز كمتجه كثيف (Dense Vector) في فضاء مستمر، بحيث **الكلمات ذات المعنى المتشابه تكون متجهاتها قريبة من بعضها**. هذا هو حجر الأساس الذي بُنيت عليه كل الثورة الحديثة في معالجة اللغات الطبيعية.


## 2. الخريطة الذهنية


```text
تطوّر تمثيل الكلمات:

One-Hot Encoding        →    Word2Vec/GloVe        →    Contextual Embeddings
(2013 وقبله)                 (Static, 2013-2017)        (BERT/GPT وما بعده، 2018+)
     │                            │                            │
متجه متناثر ضخم           متجه كثيف ثابت              متجه كثيف يتغيّر
بُعد = حجم المفردات        لكل كلمة بغض النظر          حسب السياق المحيط
لا معنى دلالي              عن السياق                   (نفس الكلمة، متجهات مختلفة!)
                           "بنك" له متجه واحد فقط       "بنك" (مالي) ≠ "بنك" (نهر)
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس — الفرضية التوزيعية (Distributional Hypothesis)

المبدأ الأساسي وراء كل Word Embeddings: **"الكلمة تُعرَف من خلال رفقتها"** (You shall know a word by the company it keeps — J.R. Firth، 1957). كلمتان تظهران غالبًا في سياقات متشابهة (محاطتان بنفس الكلمات المجاورة) من المرجّح أن تحملا معنى متشابهًا. Word2Vec يُحوِّل هذه الفرضية اللغوية إلى **مهمة تنبؤية قابلة للتدريب**: درّب شبكة عصبية بسيطة للتنبؤ بالكلمات المجاورة انطلاقًا من كلمة معينة (أو العكس)، والتمثيلات الوسيطة المُتعلَّمة أثناء هذا التدريب تصبح هي "الـ Embeddings" المفيدة.

### 🟡 طبقة الآلية — معماريتا Word2Vec


```text
CBOW (Continuous Bag of Words)         Skip-gram
"تنبأ بالكلمة من السياق"                "تنبأ بالسياق من الكلمة"

السياق: [القطة, تحب, ___, اللذيذ]        الكلمة: الحليب
         → تنبأ بـ "الحليب"              → تنبأ بـ [القطة, تحب, اللذيذ]

مناسب لـ: مجموعات بيانات كبيرة،          مناسب لـ: الكلمات النادرة،
كلمات شائعة، تدريب أسرع                  يُعطي تمثيلات أدق لكلمات نادرة
```


### 🔴 طبقة التعمّق — Skip-gram مع Negative Sampling من الصفر


In [ ]:
import numpy as np
from collections import Counter
import random

class Word2VecSkipGram:
    """
    تطبيق Skip-gram مع Negative Sampling (Mikolov et al., 2013)
    من الصفر بـ NumPy. هذا هو التطبيق الأصلي الذي أحدث ثورة في
    تمثيل الكلمات قبل ظهور المحوّلات.
    """
    def __init__(self, vocab_size: int, embedding_dim: int = 100,
                 learning_rate: float = 0.025):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.lr = learning_rate

        # مصفوفتا تضمين منفصلتان: واحدة للكلمة "المركزية" وأخرى للسياق
        # (هذا تفصيل تنفيذي مهم: نحتفظ بتمثيلين لكل كلمة أثناء التدريب،
        # ونستخدم عادة مصفوفة الكلمة المركزية فقط كـ Embeddings النهائية)
        self.W_center = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim
        self.W_context = np.zeros((vocab_size, embedding_dim))

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def train_pair(self, center_idx: int, context_idx: int,
                    negative_indices: list):
        """
        خطوة تدريب واحدة: زوج (كلمة مركزية، كلمة سياق حقيقية)
        + عدة عينات سلبية (كلمات عشوائية ليست في السياق الفعلي).

        الهدف: نُعظِّم P(context_idx|center_idx) = σ(v_center · v_context)
        ونُصغِّر P(negative|center_idx) لكل عينة سلبية — بدلًا من
        حساب softmax كامل عبر كل المفردات (مُكلِف جدًا حسابيًا
        لمفردات بحجم عشرات الآلاف أو أكثر).
        """
        v_center = self.W_center[center_idx]

        # === العينة الإيجابية (السياق الحقيقي) ===
        v_context_pos = self.W_context[context_idx]
        score_pos = self.sigmoid(np.dot(v_center, v_context_pos))
        # الخطأ: نريد score_pos قريبًا من 1 (تصنيف "إيجابي" صحيح)
        grad_pos = score_pos - 1

        grad_center_total = grad_pos * v_context_pos
        self.W_context[context_idx] -= self.lr * grad_pos * v_center

        # === العينات السلبية (كلمات عشوائية، نريد score قريبًا من 0) ===
        for neg_idx in negative_indices:
            v_context_neg = self.W_context[neg_idx]
            score_neg = self.sigmoid(np.dot(v_center, v_context_neg))
            grad_neg = score_neg - 0  # نريد قريبًا من 0

            grad_center_total += grad_neg * v_context_neg
            self.W_context[neg_idx] -= self.lr * grad_neg * v_center

        # تحديث متجه الكلمة المركزية بتجميع كل التدرجات
        self.W_center[center_idx] -= self.lr * grad_center_total

        # حساب الخسارة للمراقبة (Negative Log-Likelihood)
        loss = -np.log(score_pos + 1e-10)
        for neg_idx in negative_indices:
            score_neg = self.sigmoid(np.dot(v_center, self.W_context[neg_idx]))
            loss -= np.log(1 - score_neg + 1e-10)
        return loss

class NegativeSampler:
    """
    اختيار عينات سلبية بناءً على توزيع تكرار الكلمات، لكن مع
    "تنعيم" (smoothing) الأس 3/4 — حيلة مهمة جدًا من الورقة الأصلية
    لمنع الكلمات شائعة جدًا (مثل أدوات الربط) من الهيمنة الكاملة
    على العينات السلبية، مع إعطاء الكلمات النادرة فرصة أكبر نسبيًا
    من توزيعها الخام.
    """
    def __init__(self, word_frequencies: dict, power: float = 0.75):
        words = list(word_frequencies.keys())
        freqs = np.array([word_frequencies[w] for w in words], dtype=np.float64)

        # التنعيم: رفع كل تكرار للأس 0.75 ثم إعادة التطبيع
        smoothed_freqs = freqs ** power
        self.probabilities = smoothed_freqs / smoothed_freqs.sum()
        self.words = words

    def sample(self, n: int, exclude: set = None) -> list:
        samples = []
        while len(samples) < n:
            candidate = np.random.choice(len(self.words), p=self.probabilities)
            if exclude is None or candidate not in exclude:
                samples.append(candidate)
        return samples

def demonstrate_smoothing_effect():
    """توضيح أثر الأس 3/4 عمليًا: تقليل هيمنة الكلمات الشائعة جدًا."""
    word_freqs = {"ال": 10000, "في": 5000, "كتاب": 50, "يراعة": 2}

    freqs = np.array(list(word_freqs.values()), dtype=np.float64)
    raw_probs = freqs / freqs.sum()
    smoothed_probs = (freqs ** 0.75) / (freqs ** 0.75).sum()

    for word, raw, smooth in zip(word_freqs.keys(), raw_probs, smoothed_probs):
        print(f"{word}: احتمال خام={raw:.4f}, احتمال مُنعَّم={smooth:.4f}")

demonstrate_smoothing_effect()
# ال: احتمال خام=0.6645, احتمال مُنعَّم=0.4850    <- انخفضت هيمنته
# في: احتمال خام=0.3322, احتمال مُنعَّم=0.3630
# كتاب: احتمال خام=0.0033, احتمال مُنعَّم=0.0125   <- زادت فرصته نسبيًا
# يراعة: احتمال خام=0.0001, احتمال مُنعَّم=0.0019   <- زادت فرصته بشكل ملحوظ نسبيًا


### GloVe — نهج إحصائي بديل


In [ ]:
def explain_glove_concept():
    """
    GloVe (Global Vectors, Pennington et al., 2014): بدلًا من
    التدريب التنبؤي المحلي (نافذة سياق صغيرة كما في Word2Vec)،
    يبني GloVe أولًا "مصفوفة تعايش عامة" (co-occurrence matrix)
    X حيث X[i][j] = عدد مرات ظهور الكلمة j في سياق الكلمة i
    عبر **المجموعة النصية بأكملها**، ثم يُدرِّب Embeddings بحيث:

        v_i · v_j + b_i + b_j ≈ log(X[i][j])

    الفكرة الجوهرية: نسب تعايش الكلمات (وليس القيم الخام فقط)
    تحمل معلومات دلالية قوية جدًا. مثال كلاسيكي من الورقة:

        P(ice|solid) / P(steam|solid)  >> 1    (الجليد صلب فعلًا)
        P(ice|gas) / P(steam|gas)      << 1    (البخار غاز فعلًا)
        P(ice|water) / P(steam|water)  ≈ 1     (كلاهما متعلق بالماء)

    GloVe يستغل هذه النسب مباشرة في دالة الخسارة، بعكس Word2Vec
    الذي يتعلّمها بشكل ضمني فقط عبر التنبؤ المحلي المتكرر.
    """
    pass

def weighted_least_squares_glove_loss(co_occurrence_matrix, word_vectors,
                                        context_vectors, biases, context_biases,
                                        x_max=100, alpha=0.75):
    """
    دالة خسارة GloVe الكاملة: Weighted Least Squares.
    الوزن f(X_ij) يمنع الأزواج النادرة جدًا (ضوضاء إحصائية) والمتكررة
    جدًا (مهيمنة بشكل غير مفيد) من التأثير المفرط على التدريب.
    """
    total_loss = 0.0
    rows, cols = co_occurrence_matrix.nonzero()

    for i, j in zip(rows, cols):
        x_ij = co_occurrence_matrix[i, j]

        # دالة الوزن f(x): تتصاعد حتى x_max ثم تستقر عند 1
        weight = (x_ij / x_max) ** alpha if x_ij < x_max else 1.0

        prediction = (np.dot(word_vectors[i], context_vectors[j]) +
                      biases[i] + context_biases[j])
        target = np.log(x_ij + 1e-10)

        total_loss += weight * (prediction - target) ** 2

    return total_loss


### Static مقابل Contextual Embeddings — التوضيح الحاسم


In [ ]:
def demonstrate_static_vs_contextual_limitation():
    """
    القيد الجوهري لـ Word2Vec/GloVe (Static Embeddings):
    لكل كلمة متجه واحد ثابت بغض النظر عن السياق، لذا لا يمكنها
    التمييز بين المعاني المتعددة لنفس الكلمة (Polysemy).
    """
    sentences_with_bank = [
        "ذهبت إلى البنك لسحب المال",          # بنك = مؤسسة مالية
        "جلسنا على ضفة النهر (بنك النهر)",     # بنك = ضفة (استخدام قديم/مجازي)
    ]

    print("مع Word2Vec/GloVe (Static):")
    print("  كلمة 'بنك' → نفس المتجه بالضبط في كلا الجملتين")
    print("  (النموذج لا يستطيع التمييز بين المعنيين إطلاقًا)")

    print("\nمع BERT/GPT (Contextual، الفصل 12 لاحقًا):")
    print("  كلمة 'بنك' في الجملة 1 → متجه قريب من [مال، حساب، قرض]")
    print("  كلمة 'بنك' في الجملة 2 → متجه قريب من [نهر، ماء، ضفة]")
    print("  (نفس الرمز، تمثيلان مختلفان تمامًا حسب السياق المحيط)")

demonstrate_static_vs_contextual_limitation()


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: احسب يدويًا خطوة تدريب Skip-gram واحدة لجملة عربية، موضحًا كيف يتغيّر المتجه المركزي.


In [ ]:
def manual_skipgram_step():
    vocab = {"القطة": 0, "تحب": 1, "الحليب": 2, "اللذيذ": 3, "السيارة": 4}
    vocab_size, embed_dim = len(vocab), 4

    np.random.seed(0)
    model = Word2VecSkipGram(vocab_size, embed_dim, learning_rate=0.1)

    # الجملة: "القطة تحب الحليب اللذيذ"
    # مع نافذة سياق = 1: (تحب → القطة), (تحب → الحليب)
    center_word = "تحب"
    context_word = "الحليب"
    negative_word = "السيارة"  # كلمة غير ذات صلة، عينة سلبية

    center_idx = vocab[center_word]
    context_idx = vocab[context_word]
    negative_idx = vocab[negative_word]

    v_center_before = model.W_center[center_idx].copy()

    loss = model.train_pair(center_idx, context_idx, [negative_idx])

    v_center_after = model.W_center[center_idx]
    change = np.linalg.norm(v_center_after - v_center_before)

    print(f"الخسارة: {loss:.4f}")
    print(f"مقدار التغيير في متجه '{center_word}': {change:.6f}")
    print(f"المتجه قبل: {v_center_before.round(4)}")
    print(f"المتجه بعد: {v_center_after.round(4)}")

manual_skipgram_step()


---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| استخدم Negative Sampling بدلًا من softmax الكامل | يُقلِّل التعقيد الحسابي من O(V) إلى O(k) لكل خطوة تدريب، حيث k≈5-20 |
| اختر نافذة سياق مناسبة | نافذة صغيرة (2-5) تلتقط علاقات نحوية، نافذة كبيرة (10+) تلتقط علاقات موضوعية/دلالية عامة |
| استخدم Subsampling للكلمات شائعة جدًا | حذف عشوائي لجزء من ظهورات الكلمات شائعة جدًا (مثل "ال", "في") يُحسِّن جودة Embeddings للكلمات الأقل شيوعًا |
| قِس جودة Embeddings بمهام تناظر (Analogy) | `ملك - رجل + امرأة ≈ ملكة` اختبار كلاسيكي وسريع لجودة الفضاء الدلالي المُتعلَّم |


## 6. أخطاء شائعة ومصائد

- **استخدام Static Embeddings لمهام تتطلب فهم السياق الدقيق**: تحليل المشاعر أو الإجابة على الأسئلة تحتاج Contextual Embeddings (الفصل 12)، وStatic Embeddings وحدها غير كافية لهذه المهام المعقدة.
- **إهمال تنعيم الأس 3/4 في Negative Sampling**: يؤدي لهيمنة الكلمات شائعة الاستخدام على العينات السلبية، مما يُضعف جودة تمثيل الكلمات النادرة.
- **الخلط بين حجم الـ Embedding وجودته**: زيادة `embedding_dim` بلا حدود لا تُحسِّن الجودة دائمًا، وقد تُسبب overfitting مع بيانات محدودة.


## 7. جدول مقارنة

| الخاصية | Word2Vec | GloVe |
|---|---|---|
| منهجية التدريب | تنبؤية محلية (نافذة سياق) | إحصائية عامة (مصفوفة تعايش كاملة) |
| كفاءة استغلال البيانات | يحتاج مرور متكرر على السياقات | يستغل إحصائيات التعايش الكاملة مباشرة |
| قابلية التوسع لمجموعات ضخمة | جيدة (تدريب تدريجي/stream) | يحتاج بناء مصفوفة تعايش كاملة مسبقًا |
| الأداء التجريبي | ممتاز، خصوصًا Skip-gram للكلمات النادرة | ممتاز أيضًا، أحيانًا أفضل في مهام تناظر عامة |


## 8. ملخص أفضل الممارسات

- استخدم Skip-gram مع Negative Sampling كخط أساس قوي ومُثبَت لتعلّم Embeddings ثابتة.
- طبّق تنعيم الأس 3/4 دائمًا في اختيار العينات السلبية.
- تذكّر أن Static Embeddings (Word2Vec/GloVe) تبقى مفيدة اليوم كخط أساس سريع أو لتهيئة أولية (initialization)، لكن Contextual Embeddings (BERT/GPT) هي المعيار لكل تطبيق حديث جاد.
- قِس جودة الـ Embeddings دائمًا بمهام تناظر أو تشابه قبل استخدامها في تطبيق فعلي.


## 9. أسئلة مقابلات

**س: لماذا نستخدم Negative Sampling بدلًا من softmax الكامل في تدريب Word2Vec؟**

**ج**: softmax الكامل يتطلب حساب مجموع الأسس (normalization term) عبر **كل** كلمات المفردات (قد تكون مئات الآلاف)، في كل خطوة تدريب واحدة — تكلفة حسابية O(V) لكل عينة، وهذا غير عملي إطلاقًا لمجموعات بيانات ضخمة. Negative Sampling يُحوِّل المسألة إلى عدة مسائل تصنيف ثنائي بسيطة ومستقلة: "هل هذا الزوج (كلمة مركزية، كلمة سياق) حقيقي أم عشوائي؟"، فنحسب فقط k+1 دوال sigmoid (حيث k≈5-20 عينة سلبية) بدلًا من V دالة أس كاملة، مما يُقلِّل التعقيد الحسابي بشكل جذري مع الحفاظ على جودة تعلّم مقاربة جدًا في الممارسة العملية.

**س: ما سبب استخدام الأس 3/4 (وليس 1) في توزيع أخذ العينات السلبية؟**

**ج**: التوزيع الخام لتكرار الكلمات في أي لغة طبيعية شديد الانحراف (Zipfian): كلمات قليلة جدًا (أدوات ربط، ضمائر) تهيمن على معظم الظهورات. إذا اخترنا عينات سلبية بناءً على هذا التوزيع الخام مباشرة، ستكون العينات السلبية دائمًا تقريبًا نفس الكلمات الشائعة جدًا، مما يُضعف تعلّم تمييز دلالي دقيق للكلمات الأقل شيوعًا (لأن النموذج "يتعلّم فقط" تمييز الكلمة المركزية عن كلمات شائعة جدًا، وليس عن منافسين دلاليين أقرب). رفع التكرارات للأس 0.75 "يُسطِّح" التوزيع جزئيًا: يُقلِّل الاحتمال النسبي للكلمات شائعة جدًا بينما يزيد الاحتمال النسبي للكلمات متوسطة ونادرة الشيوع، مما يُحسِّن جودة العينات السلبية وبالتالي جودة Embeddings النهائية — وهذه القيمة (0.75) وُجدت تجريبيًا في الورقة الأصلية كالأفضل أداءً.


## 10. تمرين عملي

نفّذ `Word2VecSkipGram` كاملًا، ودرّبه على مجموعة نصوص عربية (يمكن استخدام مجموعة فقرات من ويكيبيديا العربية، 10,000+ كلمة)، ثم اختبر جودة الـ Embeddings المُتعلَّمة عبر حساب التشابه الكوني (من الفصل 2) بين كلمات متوقع تشابهها دلاليًا (مثل "ملك" و"أمير") ومقارنتها بكلمات غير مرتبطة.


## 11. مراجع للتعمّق

- Mikolov et al. (2013) — *Efficient Estimation of Word Representations in Vector Space* (Word2Vec)
- Mikolov et al. (2013) — *Distributed Representations of Words and Phrases and their Compositionality* (Negative Sampling)
- Pennington, Socher & Manning (2014) — *GloVe: Global Vectors for Word Representation*
- Firth (1957) — *A Synopsis of Linguistic Theory* (الفرضية التوزيعية الأصلية)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التمثيل اللغوي / الفصل 7: التمثيلات المتجهية للكلمات: Word2Vec وGloVe.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
